# Tiny-ViT: training notebook (Kaggle & Colab)

Train the from-scratch Vision Transformer with a **free Kaggle or Colab GPU**,
on synthetic shapes, CIFAR-10 or PlantVillage leaf diseases. The notebook is
**self-contained**: section 2 writes the `tiny_vit` package (identical to the
repo's `src/`), and training goes through the same `train()` function as the
`vit-train` command.

| | Kaggle | Colab |
|---|---|---|
| GPU | *Settings → Accelerator → GPU T4 x2* (or P100) | *Runtime → Change runtime type → T4 GPU* |
| Internet | *Settings → Internet → On* (to download data) | on by default |
| Checkpoints kept in | `/kaggle/working` (saved with "Save Version") | Google Drive if `USE_GOOGLE_DRIVE = True` |

**Sections:** 1. The package · 2. Settings · 3. Data · 4. Model ·
5. Training with a live dashboard · 6. Results · 7. Download

## 1. The package

The cells below write `src/tiny_vit/*.py`, copied from the repo at build time.
Read them in this order: `config` → `data` → `model` → `train`.

In [ ]:
import os

os.makedirs("src/tiny_vit", exist_ok=True)

In [ ]:
%%writefile src/tiny_vit/__init__.py
"""Tiny-ViT: a Vision Transformer image classifier written from scratch in PyTorch."""

from tiny_vit.config import DataConfig, ModelConfig, TrainConfig
from tiny_vit.model import VisionTransformer

__all__ = ["DataConfig", "ModelConfig", "TrainConfig", "VisionTransformer"]
__version__ = "0.2.0"

In [ ]:
%%writefile src/tiny_vit/config.py
"""Every hyperparameter of the project, in three small dataclasses.

DataConfig   what images we read and how we prepare them
ModelConfig  the shape of the Vision Transformer
TrainConfig  how we optimise it

The CLIs (`vit-train`, ...) build these from flags; checkpoints store them as
plain dicts so a saved model can always be rebuilt exactly.
"""

from __future__ import annotations

import dataclasses
from dataclasses import dataclass
from typing import Any

# ImageNet statistics: the usual choice, and close enough for most photos.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DATASETS = ("shapes", "cifar10", "folder")
AUGMENTATIONS = ("none", "light", "strong")
PATCH_EMBEDDINGS = ("linear", "conv")
POOLINGS = ("cls", "mean")

# The 12 tomato / potato classes of PlantVillage the first version of this
# project was trained on (folder names of the PlantVillage "color" release).
PLANTVILLAGE_TOMATO_POTATO = (
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
)


@dataclass
class DataConfig:
    """Where the images come from and how they are turned into tensors."""

    # "shapes": synthetic, generated on the fly (no download)
    # "cifar10": the classic 10-class 32x32 benchmark (downloaded by torchvision)
    # "folder": any folder laid out as data_dir/<class name>/<image files>
    dataset: str = "folder"
    data_dir: str = "data/plantvillage"
    # Only these class folders (None = every sub-folder of data_dir).
    classes: tuple[str, ...] | None = None
    # Cap on images per class (None = all). Handy for quick experiments.
    max_per_class: int | None = None
    # Fractions of each class held out (folder / shapes). CIFAR-10 has its own test set.
    val_fraction: float = 0.1
    test_fraction: float = 0.1
    split_seed: int = 0  # the split depends only on this, never on the training seed
    image_size: int = 128
    augment: str = "light"
    image_mean: tuple[float, float, float] = IMAGENET_MEAN
    image_std: tuple[float, float, float] = IMAGENET_STD
    shapes_per_class: int = 2000  # synthetic images per class for "shapes"

    def __post_init__(self) -> None:
        if self.dataset not in DATASETS:
            raise ValueError(f"dataset must be one of {DATASETS}, got {self.dataset!r}")
        if self.augment not in AUGMENTATIONS:
            raise ValueError(f"augment must be one of {AUGMENTATIONS}, got {self.augment!r}")
        if not (0 <= self.val_fraction < 1 and 0 <= self.test_fraction < 1):
            raise ValueError("val_fraction and test_fraction must be in [0, 1)")
        if self.val_fraction + self.test_fraction >= 1:
            raise ValueError("val_fraction + test_fraction must leave some training data")
        if self.classes is not None:
            self.classes = tuple(self.classes)  # lists come back from checkpoints
        self.image_mean = tuple(self.image_mean)
        self.image_std = tuple(self.image_std)


@dataclass
class ModelConfig:
    """Shape of the Vision Transformer.

    The image is cut into (image_size / patch_size)^2 square patches; each patch
    becomes one token of width d_model.
    """

    num_classes: int
    image_size: int = 128
    patch_size: int = 16
    in_channels: int = 3
    d_model: int = 256  # token width
    n_layers: int = 6  # Transformer blocks
    n_heads: int = 8  # attention heads (head size = d_model / n_heads)
    mlp_ratio: float = 4.0  # hidden width of the MLP = mlp_ratio * d_model
    dropout: float = 0.1
    # "linear": flatten each patch, LayerNorm, Linear (the first version's default)
    # "conv":   one Conv2d with kernel = stride = patch_size (same maths, one op)
    patch_embed: str = "linear"
    # "cls": classify from the extra [CLS] token (original ViT)
    # "mean": classify from the average of the patch tokens
    pool: str = "cls"

    def __post_init__(self) -> None:
        if self.num_classes < 1:
            raise ValueError("num_classes must be >= 1")
        if self.image_size % self.patch_size != 0:
            raise ValueError(
                f"image_size ({self.image_size}) must be a multiple of patch_size "
                f"({self.patch_size}), otherwise border pixels are silently dropped"
            )
        if self.d_model % self.n_heads != 0:
            raise ValueError(
                f"d_model ({self.d_model}) must be divisible by n_heads ({self.n_heads})"
            )
        if self.patch_embed not in PATCH_EMBEDDINGS:
            raise ValueError(f"patch_embed must be one of {PATCH_EMBEDDINGS}")
        if self.pool not in POOLINGS:
            raise ValueError(f"pool must be one of {POOLINGS}")

    @property
    def num_patches(self) -> int:
        return (self.image_size // self.patch_size) ** 2

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads


@dataclass
class TrainConfig:
    """How the model is optimised."""

    output_dir: str = "checkpoints/run"
    epochs: int = 30
    batch_size: int = 64  # per process (per GPU)
    grad_accum_steps: int = 1  # effective batch = batch_size * grad_accum_steps * GPUs
    lr: float = 5e-4  # peak learning rate, reached after warmup
    weight_decay: float = 0.05
    warmup_steps: int = 500
    label_smoothing: float = 0.1
    grad_clip: float = 1.0  # max gradient norm (0 = off)
    precision: str = "auto"  # auto | fp32 | bf16 | fp16
    device: str = "auto"  # auto | cpu | cuda | mps
    num_workers: int = 4
    seed: int = 0
    log_every: int = 50  # optimizer steps between log lines
    eval_every: int = 0  # optimizer steps between validations (0 = once per epoch)
    resume: str | None = None  # checkpoint to continue from
    overfit: int | None = None  # train AND validate on the first N training images

    def __post_init__(self) -> None:
        if self.epochs < 1 or self.batch_size < 1 or self.grad_accum_steps < 1:
            raise ValueError("epochs, batch_size and grad_accum_steps must be >= 1")
        if self.precision not in ("auto", "fp32", "bf16", "fp16"):
            raise ValueError(f"unknown precision {self.precision!r}")
        if not 0 <= self.label_smoothing < 1:
            raise ValueError("label_smoothing must be in [0, 1)")


# Sensible defaults per dataset, used by the CLI when a flag is not given.
DATASET_PRESETS: dict[str, dict[str, Any]] = {
    # 32x32 images, 4x4 patches -> 64 tokens. Small enough for a laptop CPU.
    "shapes": {"image_size": 32, "patch_size": 4, "d_model": 128, "n_layers": 4, "n_heads": 4},
    "cifar10": {"image_size": 32, "patch_size": 4, "d_model": 192, "n_layers": 6, "n_heads": 6},
    # Leaf photos are 256x256; 128 px with 16 px patches -> 64 tokens.
    "folder": {"image_size": 128, "patch_size": 16, "d_model": 256, "n_layers": 6, "n_heads": 8},
}


def config_to_dict(config: Any) -> dict[str, Any]:
    """Dataclass -> plain dict (only built-in types: safe for torch.load(weights_only=True))."""
    return dataclasses.asdict(config)


def config_from_dict(cls: type, values: dict[str, Any]) -> Any:
    """Inverse of config_to_dict. Unknown keys are ignored (forward compatibility)."""
    names = {f.name for f in dataclasses.fields(cls)}
    return cls(**{k: v for k, v in values.items() if k in names})


__all__ = [
    "DATASETS",
    "DATASET_PRESETS",
    "IMAGENET_MEAN",
    "IMAGENET_STD",
    "PLANTVILLAGE_TOMATO_POTATO",
    "DataConfig",
    "ModelConfig",
    "TrainConfig",
    "config_from_dict",
    "config_to_dict",
]

In [ ]:
%%writefile src/tiny_vit/data.py
"""Datasets and image transforms.

Three sources, one interface (`build_datasets` -> train / val / test + class names):

- "shapes":  synthetic images of 6 shapes in random colours, sizes and positions,
             generated on the fly (no download). The class depends only on the
             shape, so a model that scores well must have read the pixels.
- "cifar10": the classic 10-class 32x32 benchmark (downloaded by torchvision).
- "folder":  any directory laid out as data_dir/<class name>/<image files>, e.g.
             PlantVillage (see docs/datasets.md for the download command).

Splits are stratified (same class balance in train / val / test) and depend
only on DataConfig.split_seed, so evaluation always sees the same test images.
"""

from __future__ import annotations

import random
from collections import defaultdict
from collections.abc import Callable, Sequence
from dataclasses import dataclass
from pathlib import Path

import torch
from PIL import Image, ImageDraw
from torch.utils.data import Dataset
from torchvision import transforms

from tiny_vit.config import DataConfig

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# ---------------------------------------------------------------------------
# 1. Transforms
# ---------------------------------------------------------------------------


def build_transform(config: DataConfig, train: bool) -> Callable[[Image.Image], torch.Tensor]:
    """PIL image -> normalised (3, image_size, image_size) tensor.

    Evaluation always uses the plain resize. Training adds augmentation:
    "light"  random crop (70-100% of the image) + horizontal flip
    "strong" also vertical flip, TrivialAugment and random erasing: ViTs have
             no built-in image prior, so from scratch they need a lot of it.
    """
    size = config.image_size
    normalize = [transforms.ToTensor(), transforms.Normalize(config.image_mean, config.image_std)]
    if not train or config.augment == "none":
        return transforms.Compose([transforms.Resize((size, size)), *normalize])
    if config.augment == "light":
        return transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.7, 1.0), antialias=True),
            transforms.RandomHorizontalFlip(),
            *normalize,
        ])  # fmt: skip
    return transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.5, 1.0), antialias=True),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.TrivialAugmentWide(),
        *normalize,
        transforms.RandomErasing(p=0.25),
    ])  # fmt: skip


def denormalize(pixels: torch.Tensor, config: DataConfig) -> torch.Tensor:
    """Undo Normalize: (3, H, W) model input -> values in [0, 1] for display."""
    mean = torch.tensor(config.image_mean).view(3, 1, 1)
    std = torch.tensor(config.image_std).view(3, 1, 1)
    return (pixels.cpu() * std + mean).clamp(0, 1)


# ---------------------------------------------------------------------------
# 2. One dataset class for every source
# ---------------------------------------------------------------------------


class ImageDataset(Dataset):
    """A list of (key, label) pairs; `load(key)` returns the PIL image.

    The key is a file path (folder), an index (shapes, CIFAR-10): the loading
    differs, the rest is shared.
    """

    def __init__(
        self,
        items: Sequence[tuple[object, int]],
        load: Callable[[object], Image.Image],
        transform: Callable[[Image.Image], torch.Tensor],
    ) -> None:
        self.items = list(items)
        self.load = load
        self.transform = transform

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        key, label = self.items[index]
        return self.transform(self.load(key).convert("RGB")), label

    @property
    def labels(self) -> list[int]:
        return [label for _, label in self.items]


def open_image(path: object) -> Image.Image:
    with Image.open(path) as image:
        return image.convert("RGB")  # convert() reads the pixels before the file closes


def stratified_split(
    items: Sequence[tuple[object, int]],
    fractions: Sequence[float],
    seed: int,
) -> list[list[tuple[object, int]]]:
    """Split items into len(fractions) + 1 parts with the same class balance.

    fractions are the held-out parts (e.g. val, test); the rest is the first
    part (train). Every class keeps at least one training item.
    """
    by_class: dict[int, list] = defaultdict(list)
    for item in items:
        by_class[item[1]].append(item)
    rng = random.Random(seed)
    parts: list[list] = [[] for _ in range(len(fractions) + 1)]
    for label in sorted(by_class):
        group = by_class[label][:]
        rng.shuffle(group)
        start = 0
        for i, fraction in enumerate(fractions, start=1):
            n = min(round(fraction * len(group)), len(group) - start - 1)
            parts[i].extend(group[start : start + max(n, 0)])
            start += max(n, 0)
        parts[0].extend(group[start:])
    return parts


def balanced_subset(items: Sequence[tuple[object, int]], n: int) -> list[tuple[object, int]]:
    """The first n items taken round-robin over the classes (class 0, 1, 2, 0, 1, ...)."""
    by_class: dict[int, list] = defaultdict(list)
    for item in items:
        by_class[item[1]].append(item)
    queues = [by_class[label] for label in sorted(by_class)]
    subset: list[tuple[object, int]] = []
    for i in range(max(map(len, queues), default=0)):
        subset.extend(queue[i] for queue in queues if i < len(queue))
    return subset[:n]


# ---------------------------------------------------------------------------
# 3. Sources
# ---------------------------------------------------------------------------

# --- synthetic shapes ---

SHAPE_CLASSES = ("circle", "square", "triangle", "diamond", "cross", "ring")
SHAPE_CANVAS = 64  # rendered at 64x64; the transform resizes to image_size
_COLORS = [
    (230, 60, 60), (60, 180, 75), (60, 110, 230), (255, 200, 40),
    (240, 130, 40), (160, 80, 200), (70, 210, 210), (240, 80, 180),
]  # fmt: skip


class ShapeRenderer:
    """Draws image `index` deterministically: same (seed, index) -> same picture."""

    def __init__(self, seed: int) -> None:
        self.seed = seed

    def __call__(self, index: object) -> Image.Image:
        index = int(index)
        rng = random.Random(self.seed * 1_000_003 + index)
        kind = SHAPE_CLASSES[index % len(SHAPE_CLASSES)]
        shade = rng.randint(0, 60) if rng.random() < 0.5 else rng.randint(190, 255)
        background = tuple(min(255, max(0, shade + rng.randint(-15, 15))) for _ in range(3))
        image = Image.new("RGB", (SHAPE_CANVAS, SHAPE_CANVAS), background)
        draw = ImageDraw.Draw(image)
        r = rng.uniform(9, 22)  # half-size of the shape
        cx = rng.uniform(r + 2, SHAPE_CANVAS - r - 2)
        cy = rng.uniform(r + 2, SHAPE_CANVAS - r - 2)
        color = rng.choice(_COLORS)
        box = (cx - r, cy - r, cx + r, cy + r)
        if kind == "circle":
            draw.ellipse(box, fill=color)
        elif kind == "ring":
            draw.ellipse(box, outline=color, width=max(2, round(r / 3)))
        elif kind == "square":
            draw.rectangle(box, fill=color)
        elif kind == "triangle":
            draw.polygon([(cx, cy - r), (cx + r, cy + r), (cx - r, cy + r)], fill=color)
        elif kind == "diamond":
            draw.polygon([(cx, cy - r), (cx + r, cy), (cx, cy + r), (cx - r, cy)], fill=color)
        else:  # cross
            w = r / 2.5
            draw.rectangle((cx - w, cy - r, cx + w, cy + r), fill=color)
            draw.rectangle((cx - r, cy - w, cx + r, cy + w), fill=color)
        return image


def shapes_items(config: DataConfig) -> list[tuple[object, int]]:
    total = config.shapes_per_class * len(SHAPE_CLASSES)
    return [(i, i % len(SHAPE_CLASSES)) for i in range(total)]


# --- image folder ---


def find_classes(data_dir: Path, classes: Sequence[str] | None) -> list[str]:
    """Class names = sub-folder names, sorted (or the requested subset, checked)."""
    if not data_dir.is_dir():
        raise FileNotFoundError(
            f"data directory not found: {data_dir} (expected data_dir/<class name>/<images>; "
            "see docs/datasets.md)"
        )
    available = sorted(p.name for p in data_dir.iterdir() if p.is_dir())
    if classes is None:
        if not available:
            raise FileNotFoundError(f"no class sub-folders in {data_dir}")
        return available
    missing = [c for c in classes if c not in available]
    if missing:
        raise FileNotFoundError(f"class folders not found in {data_dir}: {missing}")
    return sorted(classes)


def folder_items(
    data_dir: Path, class_names: Sequence[str], max_per_class: int | None, seed: int
) -> list[tuple[object, int]]:
    items = []
    rng = random.Random(seed)
    for label, name in enumerate(class_names):
        files = sorted(
            p for p in (data_dir / name).rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS
        )
        if not files:
            raise FileNotFoundError(f"no images in {data_dir / name}")
        if max_per_class is not None and len(files) > max_per_class:
            files = sorted(rng.sample(files, max_per_class))  # random subset, not the first N
        items.extend((str(p), label) for p in files)
    return items


# --- CIFAR-10 ---


class CifarImage:
    """Image `index` of a torchvision CIFAR-10 split (picklable, for DataLoader workers)."""

    def __init__(self, root: str, train: bool) -> None:
        from torchvision.datasets import CIFAR10

        self.dataset = CIFAR10(root, train=train, download=True)

    def __call__(self, index: object) -> Image.Image:
        return self.dataset[int(index)][0]


# ---------------------------------------------------------------------------
# 4. Entry point
# ---------------------------------------------------------------------------


@dataclass
class Datasets:
    train: ImageDataset
    val: ImageDataset
    test: ImageDataset
    class_names: list[str]


def build_datasets(config: DataConfig) -> Datasets:
    """Train / val / test datasets for the configured source."""
    train_tf, eval_tf = build_transform(config, train=True), build_transform(config, train=False)

    if config.dataset == "cifar10":
        train_source = CifarImage(config.data_dir, train=True)
        test_source = CifarImage(config.data_dir, train=False)
        class_names = list(train_source.dataset.classes)
        items = list(enumerate(train_source.dataset.targets))
        train, val = stratified_split(items, [config.val_fraction], config.split_seed)
        test = list(enumerate(test_source.dataset.targets))
        return Datasets(
            ImageDataset(train, train_source, train_tf),
            ImageDataset(val, train_source, eval_tf),
            ImageDataset(test, test_source, eval_tf),
            class_names,
        )

    if config.dataset == "shapes":
        class_names = list(SHAPE_CLASSES)
        items = shapes_items(config)
        load: Callable[[object], Image.Image] = ShapeRenderer(config.split_seed)
    else:
        data_dir = Path(config.data_dir)
        class_names = find_classes(data_dir, config.classes)
        items = folder_items(data_dir, class_names, config.max_per_class, config.split_seed)
        load = open_image

    fractions = [config.val_fraction, config.test_fraction]
    train, val, test = stratified_split(items, fractions, config.split_seed)
    return Datasets(
        ImageDataset(train, load, train_tf),
        ImageDataset(val, load, eval_tf),
        ImageDataset(test, load, eval_tf),
        class_names,
    )

In [ ]:
%%writefile src/tiny_vit/model.py
"""A Vision Transformer (ViT) written from scratch, read top to bottom.

    image (B, 3, H, W)
      -> PatchEmbedding      cut into P x P patches, one d_model vector per patch   (B, N, D)
      -> prepend [CLS] token, add learned position embeddings                       (B, N+1, D)
      -> n_layers x Block    pre-norm: x + Attention(LN(x)), then x + MLP(LN(x))     (B, N+1, D)
      -> LayerNorm, take [CLS] (or the mean of the patch tokens)                    (B, D)
      -> Linear head                                                                (B, C_out)

B = batch, N = (H / P) * (W / P) patches, D = d_model, C_out = num_classes.
Reference: Dosovitskiy et al., "An Image is Worth 16x16 Words" (ICLR 2021).
"""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import nn

from tiny_vit.config import ModelConfig

# ---------------------------------------------------------------------------
# 1. Patch embedding: image -> sequence of tokens
# ---------------------------------------------------------------------------


def patchify(images: torch.Tensor, patch_size: int) -> torch.Tensor:
    """(B, C, H, W) -> (B, N, P*P*C): each row is one flattened patch, row-major order.

    Patch n covers rows (n // W') * P ... and columns (n % W') * P ..., W' = W / P.
    """
    B, C, H, W = images.shape
    P = patch_size
    # (B, C, H/P, P, W/P, P) -> (B, H/P, W/P, P, P, C) -> (B, N, P*P*C)
    x = images.reshape(B, C, H // P, P, W // P, P)
    x = x.permute(0, 2, 4, 3, 5, 1)
    return x.reshape(B, (H // P) * (W // P), P * P * C)


class PatchEmbedding(nn.Module):
    """Turn an image into N tokens of width d_model.

    Two equivalent ways to do it (both are a linear map of each patch):
    - "linear": flatten each patch, then LayerNorm -> Linear -> LayerNorm.
      The extra LayerNorms (from the first version of this project) make
      training a little more stable when trained from scratch.
    - "conv": a Conv2d with kernel_size = stride = patch_size. Each output
      pixel of the convolution sees exactly one patch, so this is the same
      linear map, computed in one fused operation.
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.kind = config.patch_embed
        self.patch_size = config.patch_size
        patch_dim = config.patch_size**2 * config.in_channels
        if self.kind == "linear":
            self.proj = nn.Sequential(
                nn.LayerNorm(patch_dim),
                nn.Linear(patch_dim, config.d_model),
                nn.LayerNorm(config.d_model),
            )
        else:
            self.proj = nn.Conv2d(
                config.in_channels, config.d_model, config.patch_size, stride=config.patch_size
            )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if self.kind == "linear":
            return self.proj(patchify(images, self.patch_size))  # (B, N, D)
        x = self.proj(images)  # (B, D, H/P, W/P)
        return x.flatten(2).transpose(1, 2)  # (B, N, D), same patch order as patchify


# ---------------------------------------------------------------------------
# 2. Multi-head self-attention
# ---------------------------------------------------------------------------


class Attention(nn.Module):
    """Every token looks at every other token (no causal mask: images have no "future").

    Each head h computes softmax(Q_h K_h^T / sqrt(head_dim)) V_h; the heads are
    concatenated and mixed by a final linear layer.
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.n_heads = config.n_heads
        self.head_dim = config.head_dim
        self.dropout = config.dropout
        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)  # Q, K and V in one matmul
        self.out_proj = nn.Linear(config.d_model, config.d_model)
        self.out_dropout = nn.Dropout(config.dropout)

    def forward(
        self, x: torch.Tensor, return_attention: bool = False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        B, T, D = x.shape  # T = N + 1 tokens
        # (B, T, 3D) -> 3 x (B, n_heads, T, head_dim)
        q, k, v = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        # Dropout only while training: self.training is set by model.train() / model.eval().
        p = self.dropout if self.training else 0.0

        if return_attention:
            # Explicit version, so we can return the attention weights (for visualisation).
            scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, h, T, T)
            weights = scores.softmax(dim=-1)
            y = F.dropout(weights, p, self.training) @ v  # (B, h, T, head_dim)
        else:
            # Same maths, fused kernel (FlashAttention / memory-efficient on GPU).
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=p)
            weights = None

        y = y.transpose(1, 2).reshape(B, T, D)  # concatenate the heads
        return self.out_dropout(self.out_proj(y)), weights


# ---------------------------------------------------------------------------
# 3. Transformer block
# ---------------------------------------------------------------------------


class MLP(nn.Module):
    """Per-token feed-forward network: Linear -> GELU -> Linear."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        hidden = int(config.mlp_ratio * config.d_model)
        self.fc1 = nn.Linear(config.d_model, hidden)
        self.fc2 = nn.Linear(hidden, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))


class Block(nn.Module):
    """Pre-norm Transformer block: normalise, transform, add back (residual)."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = Attention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.mlp = MLP(config)

    def forward(
        self, x: torch.Tensor, return_attention: bool = False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        attended, weights = self.attn(self.ln1(x), return_attention)
        x = x + attended  # tokens exchange information
        x = x + self.mlp(self.ln2(x))  # each token is processed on its own
        return x, weights


# ---------------------------------------------------------------------------
# 4. The full model
# ---------------------------------------------------------------------------


class VisionTransformer(nn.Module):
    """Image classifier: patches -> Transformer encoder -> class logits."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.patch_embed = PatchEmbedding(config)
        # A learned token with no image content: through attention it gathers a
        # summary of the whole image, and the head classifies from it.
        self.cls_token = nn.Parameter(torch.zeros(1, 1, config.d_model))
        # Attention ignores order, so each position (CLS + every patch) gets a learned vector.
        self.pos_embed = nn.Parameter(torch.zeros(1, config.num_patches + 1, config.d_model))
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList(Block(config) for _ in range(config.n_layers))
        self.norm = nn.LayerNorm(config.d_model)
        self.head = nn.Linear(config.d_model, config.num_classes)
        self._init_weights()

    def _init_weights(self) -> None:
        """Small random weights (std 0.02) everywhere, a few deliberate exceptions."""
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Conv2d)):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        # Each block adds its output to the residual stream; scale the last layer
        # of each branch down so the stream does not grow with depth (GPT-2 trick).
        for block in self.blocks:
            for layer in (block.attn.out_proj, block.mlp.fc2):
                nn.init.trunc_normal_(layer.weight, std=0.02 / math.sqrt(2 * self.config.n_layers))
        # Zero head: every class starts equally likely, so the first loss is
        # exactly ln(num_classes) (a free sanity check, see tests/test_model.py).
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward_features(
        self, images: torch.Tensor, return_attention: bool = False
    ) -> tuple[torch.Tensor, list[torch.Tensor]]:
        """(B, C, H, W) -> normalised tokens (B, N+1, D) and (optionally) each layer's attention."""
        size = self.config.image_size
        if images.shape[-2:] != (size, size):
            raise ValueError(f"expected {size}x{size} images, got {tuple(images.shape[-2:])}")
        x = self.patch_embed(images)  # (B, N, D)
        cls = self.cls_token.expand(x.shape[0], -1, -1)  # (B, 1, D)
        x = torch.cat([cls, x], dim=1) + self.pos_embed  # (B, N+1, D)
        x = self.dropout(x)
        attentions = []
        for block in self.blocks:
            x, weights = block(x, return_attention)
            if weights is not None:
                attentions.append(weights)
        return self.norm(x), attentions

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """(B, C, H, W) -> class logits (B, num_classes)."""
        tokens, _ = self.forward_features(images)
        pooled = tokens[:, 0] if self.config.pool == "cls" else tokens[:, 1:].mean(dim=1)
        return self.head(pooled)

    @torch.no_grad()
    def attention_maps(self, images: torch.Tensor) -> list[torch.Tensor]:
        """Attention weights of every layer, each (B, n_heads, N+1, N+1)."""
        return self.forward_features(images, return_attention=True)[1]

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def parameter_groups(self, weight_decay: float) -> list[dict]:
        """Two AdamW groups: weight matrices are decayed; biases, norms,
        the [CLS] token and position embeddings are not (decaying them only hurts)."""
        decay, no_decay = [], []
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            if param.ndim >= 2 and name not in ("cls_token", "pos_embed"):
                decay.append(param)
            else:
                no_decay.append(param)
        return [
            {"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ]


def attention_rollout(attentions: list[torch.Tensor]) -> torch.Tensor:
    """Which patches did the [CLS] token look at, through all layers? (Abnar & Zuidema, 2020)

    Average the heads, add the identity for the residual connection, renormalise,
    and multiply the layers together. Returns (B, N): one weight per patch,
    summing to 1 (all 0 if [CLS] never looked at a patch). Reshape it to
    (H/P, W/P) to overlay it on the image.
    """
    rollout = None
    for weights in attentions:
        a = weights.mean(dim=1)  # (B, T, T)
        a = a + torch.eye(a.shape[-1], device=a.device)
        a = a / a.sum(dim=-1, keepdim=True)
        rollout = a if rollout is None else a @ rollout
    cls_to_patches = rollout[:, 0, 1:]
    # (clamp: if [CLS] only ever attended to itself, every patch weight is 0, not NaN)
    return cls_to_patches / cls_to_patches.sum(dim=-1, keepdim=True).clamp(min=1e-12)

In [ ]:
%%writefile src/tiny_vit/train.py
"""Train the Vision Transformer.

The file is split into small pieces you can read top to bottom:

1. setup helpers   - seeding, device, mixed precision, multi-GPU (DDP) setup
2. data            - DataLoaders
3. optimisation    - AdamW + warmup/cosine learning-rate schedule
4. evaluation      - loss and accuracy on a held-out set
5. checkpoints     - save / load everything needed to resume (or to predict)
6. train()         - the training loop (takes ready-made objects, easy to test)
7. CLI             - main() builds everything from command-line flags

Run it with:  vit-train --help

Multi-GPU / multi-node: launch the same command with torchrun, e.g.
    torchrun --nproc_per_node=4 -m tiny_vit.train ...
torchrun starts one process per GPU and sets RANK / WORLD_SIZE / LOCAL_RANK;
the code below picks those up and wraps the model in DistributedDataParallel.
"""

from __future__ import annotations

import argparse
import logging
import math
import os
import random
import time
from collections import Counter
from collections.abc import Callable, Iterator, Sequence
from contextlib import AbstractContextManager, contextmanager, nullcontext
from dataclasses import dataclass, field
from itertools import islice
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW, Optimizer
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset, DistributedSampler

from tiny_vit.config import (
    DATASET_PRESETS,
    DATASETS,
    DataConfig,
    ModelConfig,
    TrainConfig,
    config_to_dict,
)
from tiny_vit.data import ImageDataset, balanced_subset, build_datasets, build_transform
from tiny_vit.model import VisionTransformer

logger = logging.getLogger(__name__)

LAST_CHECKPOINT = "last.pt"
BEST_CHECKPOINT = "best.pt"


# ---------------------------------------------------------------------------
# 1. Setup helpers
# ---------------------------------------------------------------------------


def seed_everything(seed: int) -> None:
    """Seed every random number generator we use."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # also seeds all CUDA devices


def resolve_device(name: str = "auto") -> torch.device:
    """Turn "auto" | "cpu" | "cuda" | "mps" into a torch.device."""
    # Under torchrun each process owns one GPU: cuda:LOCAL_RANK.
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    if name == "cuda" or (name == "auto" and torch.cuda.is_available()):
        return torch.device("cuda", local_rank)
    if name != "auto":
        return torch.device(name)
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def resolve_precision(precision: str, device: torch.device) -> tuple[torch.dtype | None, bool]:
    """Pick the autocast dtype and whether we need a GradScaler.

    Returns (dtype, use_scaler). dtype None means plain fp32 (no autocast).
    - bf16 has the same range as fp32, so it needs no loss scaling.
    - fp16 has a small range; small gradients underflow to 0 without a GradScaler.
    """
    if precision == "fp32":
        return None, False
    if precision == "bf16":
        return torch.bfloat16, False
    if precision == "fp16":
        if device.type != "cuda":
            raise ValueError("fp16 training needs a CUDA device")
        return torch.float16, True
    if precision == "auto":
        if device.type != "cuda":
            return None, False  # CPU / MPS: keep it simple and exact
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16, False
        return torch.float16, True
    raise ValueError(f"unknown precision: {precision!r}")


@dataclass(frozen=True)
class DistInfo:
    """Who am I in a multi-process run? Defaults describe a single process."""

    rank: int = 0  # global id of this process (0 .. world_size-1)
    world_size: int = 1  # total number of processes (= total GPUs)
    local_rank: int = 0  # id of this process on its own machine

    @property
    def enabled(self) -> bool:
        return self.world_size > 1

    @property
    def is_main(self) -> bool:
        """Only the main process logs and writes checkpoints."""
        return self.rank == 0


def setup_distributed(device_name: str = "auto") -> DistInfo:
    """Join the process group if we were launched by torchrun; no-op otherwise."""
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    if world_size == 1:
        return DistInfo()
    local_rank = int(os.environ["LOCAL_RANK"])
    use_cuda = device_name in ("auto", "cuda") and torch.cuda.is_available()
    if use_cuda:
        torch.cuda.set_device(local_rank)  # this process only talks to its own GPU
    # NCCL is the fast backend for GPUs; gloo works on CPU (handy for testing).
    dist.init_process_group(backend="nccl" if use_cuda else "gloo")
    return DistInfo(dist.get_rank(), dist.get_world_size(), local_rank)


def get_dist_info() -> DistInfo:
    """Current process-group state (single process if none was started)."""
    if not (dist.is_available() and dist.is_initialized()):
        return DistInfo()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    return DistInfo(dist.get_rank(), dist.get_world_size(), local_rank)


@contextmanager
def local_main_first(info: DistInfo) -> Iterator[None]:
    """Let local rank 0 run the block first (e.g. the CIFAR-10 download);
    the other processes on the machine wait, then reuse the files."""
    if info.enabled and info.local_rank != 0:
        dist.barrier()
    yield
    if info.enabled and info.local_rank == 0:
        dist.barrier()


def autocast(device: torch.device, dtype: torch.dtype | None) -> AbstractContextManager:
    """Mixed-precision context; does nothing when dtype is None."""
    return torch.autocast(device_type=device.type, dtype=dtype, enabled=dtype is not None)


# ---------------------------------------------------------------------------
# 2. Data
# ---------------------------------------------------------------------------


def seed_worker(worker_id: int) -> None:
    """Give each DataLoader worker its own python/numpy seed (torch is already seeded)."""
    worker_seed = torch.initial_seed() % 2**32
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def build_dataloaders(
    train_dataset: Dataset, val_dataset: Dataset, config: TrainConfig, device: torch.device
) -> tuple[DataLoader, DataLoader]:
    """Create the train and validation DataLoaders.

    With several processes, a DistributedSampler gives each one a different
    slice of the data, so batch_size is *per process* (per GPU).
    """
    info = get_dist_info()
    workers = config.num_workers
    common = {
        "batch_size": config.batch_size,
        "num_workers": workers,
        "pin_memory": device.type == "cuda",  # faster CPU -> GPU copies
        "persistent_workers": workers > 0,  # do not restart workers every epoch
        "worker_init_fn": seed_worker if workers > 0 else None,
    }
    generator = torch.Generator()
    generator.manual_seed(config.seed + info.rank)

    if info.enabled:
        train_sampler = DistributedSampler(
            train_dataset, shuffle=True, seed=config.seed, drop_last=True
        )
        # Note: to split evenly, the sampler may repeat a few val samples.
        val_sampler = DistributedSampler(val_dataset, shuffle=False, drop_last=False)
        train_loader = DataLoader(
            train_dataset, sampler=train_sampler, drop_last=True, generator=generator, **common
        )
        val_loader = DataLoader(val_dataset, sampler=val_sampler, drop_last=False, **common)
        return train_loader, val_loader

    train_loader = DataLoader(
        train_dataset, shuffle=True, drop_last=True, generator=generator, **common
    )
    val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **common)
    return train_loader, val_loader


# ---------------------------------------------------------------------------
# 3. Optimisation
# ---------------------------------------------------------------------------


def lr_lambda(step: int, warmup_steps: int, total_steps: int) -> float:
    """LR multiplier: linear warmup to 1, then cosine decay to 0.

    `step` is the number of optimizer steps already taken (starts at 0).
    """
    if step < warmup_steps:
        return (step + 1) / warmup_steps  # never exactly 0, so step 0 still learns
    decay_steps = max(1, total_steps - warmup_steps)
    progress = min(1.0, (step - warmup_steps) / decay_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def build_optimizer(model: VisionTransformer, config: TrainConfig, device: torch.device) -> AdamW:
    """AdamW; no weight decay on biases, norms, [CLS] and position embeddings."""
    groups = model.parameter_groups(config.weight_decay)
    # fused=True runs the whole update in one CUDA kernel (faster); CUDA only.
    return AdamW(groups, lr=config.lr, betas=(0.9, 0.999), fused=device.type == "cuda")


def build_scheduler(optimizer: Optimizer, warmup_steps: int, total_steps: int) -> LambdaLR:
    """Warmup + cosine schedule. Call scheduler.step() once per optimizer step."""
    return LambdaLR(optimizer, lambda step: lr_lambda(step, warmup_steps, total_steps))


# ---------------------------------------------------------------------------
# 4. Evaluation
# ---------------------------------------------------------------------------


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    amp_dtype: torch.dtype | None = None,
) -> dict[str, float]:
    """Mean cross-entropy (no label smoothing) and top-1 accuracy over the loader.

    With several processes each one evaluates its slice; the sums are added up
    so every process gets the same, global numbers.
    """
    was_training = model.training
    model.eval()  # dropout off
    totals = torch.zeros(3, device=device)  # loss sum, correct, count
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with autocast(device, amp_dtype):
            logits = model(images)
        totals[0] += F.cross_entropy(logits.float(), labels, reduction="sum")
        totals[1] += (logits.argmax(dim=-1) == labels).sum()
        totals[2] += labels.numel()
    if get_dist_info().enabled:
        dist.all_reduce(totals)
    model.train(was_training)
    loss_sum, correct, count = totals.tolist()
    count = max(count, 1.0)
    return {"loss": loss_sum / count, "accuracy": correct / count}


# ---------------------------------------------------------------------------
# 5. Checkpoints
# ---------------------------------------------------------------------------


@dataclass
class TrainState:
    """Counters that must survive a restart."""

    step: int = 0  # optimizer steps done
    epoch: int = 0  # epoch we are in
    best_val_accuracy: float = -1.0
    history: list[dict[str, Any]] = field(default_factory=list)  # every logged event


def rng_state() -> dict[str, Any]:
    """Snapshot every RNG so a resumed run continues the same random stream."""
    name, keys, pos, has_gauss, cached = np.random.get_state()
    state: dict[str, Any] = {
        "python": random.getstate(),
        "torch": torch.get_rng_state(),
        # plain lists so torch.load(weights_only=True) can read them back
        "numpy": (name, keys.tolist(), pos, has_gauss, cached),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state: dict[str, Any]) -> None:
    """Restore what rng_state() saved."""
    random.setstate(state["python"])
    torch.set_rng_state(state["torch"])
    name, keys, pos, has_gauss, cached = state["numpy"]
    np.random.set_state((name, np.array(keys, dtype=np.uint32), pos, has_gauss, cached))
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


def save_checkpoint(
    path: Path,
    model: VisionTransformer,
    optimizer: Optimizer,
    scheduler: LambdaLR,
    scaler: torch.amp.GradScaler,
    state: TrainState,
    data_config: DataConfig,
    train_config: TrainConfig,
    class_names: Sequence[str],
) -> None:
    """Save everything needed to resume training or to run the model. Written atomically."""
    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "step": state.step,
        "epoch": state.epoch,
        "best_val_accuracy": state.best_val_accuracy,
        "history": state.history,
        "class_names": list(class_names),
        "data_config": config_to_dict(data_config),
        "model_config": config_to_dict(model.config),
        "train_config": config_to_dict(train_config),
        "rng_state": rng_state(),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, tmp_path)
    # os.replace is atomic: a crash mid-save never leaves a half-written file.
    os.replace(tmp_path, path)


def load_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: Optimizer | None = None,
    scheduler: LambdaLR | None = None,
    scaler: torch.amp.GradScaler | None = None,
) -> TrainState:
    """Load a checkpoint into the given objects and return the saved counters."""
    # weights_only=True refuses to run arbitrary pickled code (safe loading).
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model"])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if scheduler is not None:
        scheduler.load_state_dict(checkpoint["scheduler"])
    if scaler is not None and checkpoint["scaler"]:  # empty dict when scaler disabled
        scaler.load_state_dict(checkpoint["scaler"])
    set_rng_state(checkpoint["rng_state"])
    return TrainState(
        step=checkpoint["step"],
        epoch=checkpoint["epoch"],
        best_val_accuracy=checkpoint["best_val_accuracy"],
        history=list(checkpoint.get("history", [])),
    )


# ---------------------------------------------------------------------------
# 6. Training loop
# ---------------------------------------------------------------------------


def epoch_batches(loader: DataLoader, epoch: int, seed: int, skip: int) -> Iterator:
    """Batches of one epoch, in a shuffle order that depends only on (seed, epoch).

    Re-seeding per epoch means a resumed run sees the same order, so we can
    skip the batches that were already used before the restart.
    """
    if isinstance(loader.sampler, DistributedSampler):
        loader.sampler.set_epoch(epoch)  # new shuffle each epoch, same on every process
    elif loader.generator is not None:
        loader.generator.manual_seed(seed + epoch)
    return islice(iter(loader), skip, None)


def train(
    model: VisionTransformer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    data_config: DataConfig,
    train_config: TrainConfig,
    class_names: Sequence[str],
    callback: Callable[[dict[str, Any]], None] | None = None,
) -> TrainState:
    """Run the full training loop and return the final counters.

    Everything is passed in already built, so tests can use tiny datasets.
    `callback` (optional) receives every logged metric as a dict, on the main
    process only; the notebook uses it to draw live training curves:
        {"kind": "train", "step", "epoch", "total_steps", "loss", "accuracy", "lr",
         "grad_norm", "images_per_sec"}
        {"kind": "val", "step", "epoch", "loss", "accuracy"}
    """
    cfg = train_config
    info = get_dist_info()
    device = resolve_device(cfg.device)
    amp_dtype, use_scaler = resolve_precision(cfg.precision, device)
    model.to(device)
    model.train()
    raw_model = model  # the plain model: used for checkpoints and evaluation

    # One optimizer step = grad_accum_steps micro-batches.
    accum = cfg.grad_accum_steps
    steps_per_epoch = len(train_loader) // accum
    if steps_per_epoch == 0:
        raise ValueError("train set too small for batch_size * grad_accum_steps")
    total_steps = cfg.epochs * steps_per_epoch
    eval_every = cfg.eval_every or steps_per_epoch

    optimizer = build_optimizer(raw_model, cfg, device)
    scheduler = build_scheduler(optimizer, cfg.warmup_steps, total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    state = TrainState()
    if cfg.resume is not None:
        state = load_checkpoint(Path(cfg.resume), raw_model, optimizer, scheduler, scaler)
        logger.info("resumed from %s at step %d", cfg.resume, state.step)

    if info.enabled:
        # DDP averages the gradients of all processes after each backward pass.
        # It also copies rank 0's weights to everyone at start, so all begin equal.
        model = DDP(model, device_ids=[device.index] if device.type == "cuda" else None)

    def checkpoint(path: Path) -> None:
        if info.is_main:  # one writer is enough: all processes hold the same weights
            save_checkpoint(
                path, raw_model, optimizer, scheduler, scaler, state,
                data_config, cfg, class_names,
            )  # fmt: skip

    def emit(event: dict[str, Any]) -> None:
        if info.is_main:
            state.history.append(dict(event))
            if callback is not None:
                callback(event)

    def validate_and_save() -> None:
        metrics = evaluate(raw_model, val_loader, device, amp_dtype)
        logger.info(
            "val   | epoch %d | step %d | loss %.4f | acc %.4f",
            state.epoch, state.step, metrics["loss"], metrics["accuracy"],
        )  # fmt: skip
        if metrics["accuracy"] > state.best_val_accuracy:
            state.best_val_accuracy = metrics["accuracy"]
            checkpoint(output_dir / BEST_CHECKPOINT)
        checkpoint(output_dir / LAST_CHECKPOINT)
        # after the save: whoever receives this event can rely on last.pt being current
        emit({"kind": "val", "step": state.step, "epoch": state.epoch, **metrics})

    logger.info(
        "training %.2fM parameters: %d epochs x %d steps (effective batch %d = %d per process "
        "x %d accum x %d processes) on %s, amp=%s",
        raw_model.num_parameters() / 1e6, cfg.epochs, steps_per_epoch,
        cfg.batch_size * accum * info.world_size, cfg.batch_size, accum, info.world_size,
        device, amp_dtype,
    )  # fmt: skip

    # Running sums for the log line (kept on device: no GPU sync every step).
    window_loss = torch.zeros((), device=device)
    window_correct = torch.zeros((), device=device)
    window_images = 0
    window_steps = 0
    window_start = time.perf_counter()

    for epoch in range(state.step // steps_per_epoch, cfg.epochs):
        state.epoch = epoch
        done_in_epoch = state.step - epoch * steps_per_epoch  # > 0 only right after resume
        batches = epoch_batches(train_loader, epoch, cfg.seed, skip=done_in_epoch * accum)
        micro_batches = (steps_per_epoch - done_in_epoch) * accum  # drop the leftover tail

        for micro_step, (images, labels) in enumerate(islice(batches, micro_batches)):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            is_update_step = (micro_step + 1) % accum == 0
            # With DDP, skip the gradient all-reduce until the last micro-batch:
            # syncing once per optimizer step is enough and much cheaper.
            sync = model.no_sync() if info.enabled and not is_update_step else nullcontext()
            with sync:
                with autocast(device, amp_dtype):
                    logits = model(images)
                    loss = F.cross_entropy(
                        logits.float(), labels, label_smoothing=cfg.label_smoothing
                    )
                # Divide so the summed gradients equal the gradient of the mean loss.
                scaler.scale(loss / accum).backward()
            window_loss += loss.detach() / accum
            window_correct += (logits.detach().argmax(dim=-1) == labels).sum()
            window_images += labels.numel()

            if not is_update_step:
                continue  # keep accumulating gradients

            # --- one optimizer step ---
            scaler.unscale_(optimizer)  # back to real gradient values before clipping
            max_norm = cfg.grad_clip if cfg.grad_clip > 0 else math.inf
            grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            scaler.step(optimizer)  # skips the step if fp16 gradients overflowed
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            state.step += 1
            window_steps += 1

            if state.step % cfg.log_every == 0 or state.step == total_steps:
                elapsed = time.perf_counter() - window_start
                mean_loss = window_loss.item() / window_steps
                accuracy = window_correct.item() / max(window_images, 1)
                images_per_sec = window_images * info.world_size / elapsed
                lr = optimizer.param_groups[0]["lr"]
                logger.info(
                    "train | epoch %d | step %d/%d | loss %.4f | acc %.3f | lr %.2e | grad %.2f "
                    "| %.0f img/s",
                    epoch, state.step, total_steps, mean_loss, accuracy, lr,
                    grad_norm.item(), images_per_sec,
                )  # fmt: skip
                emit({
                    "kind": "train", "step": state.step, "epoch": epoch,
                    "total_steps": total_steps, "loss": mean_loss, "accuracy": accuracy,
                    "lr": lr, "grad_norm": grad_norm.item(), "images_per_sec": images_per_sec,
                })  # fmt: skip
                window_loss.zero_()
                window_correct.zero_()
                window_images = window_steps = 0
                window_start = time.perf_counter()

            if state.step % eval_every == 0 or state.step == total_steps:
                validate_and_save()
                window_start = time.perf_counter()  # don't count eval time in the speed

    state.epoch = cfg.epochs
    checkpoint(output_dir / LAST_CHECKPOINT)
    logger.info("done | best val accuracy %.4f", state.best_val_accuracy)
    return state


# ---------------------------------------------------------------------------
# 7. Command line
# ---------------------------------------------------------------------------

# Flags whose default depends on --dataset (see config.DATASET_PRESETS).
PRESET_FLAGS = ("image_size", "patch_size", "d_model", "n_layers", "n_heads")


class _HelpFormatter(argparse.ArgumentDefaultsHelpFormatter):
    """Show "(default: X)" on every flag, except when there is no default."""

    def _get_help_string(self, action: argparse.Action) -> str:
        if action.default is None:
            return action.help or ""
        return super()._get_help_string(action) or ""


def build_parser() -> argparse.ArgumentParser:
    """All CLI flags. Defaults come from the config dataclasses."""
    d, t, m = DataConfig(), TrainConfig(), ModelConfig(num_classes=1)
    p = argparse.ArgumentParser(
        description="Train a Vision Transformer from scratch.", formatter_class=_HelpFormatter
    )
    g = p.add_argument_group("data")
    g.add_argument("--dataset", choices=DATASETS, default="shapes", help="image source")
    g.add_argument("--data-dir", help="dataset folder (default: data/<dataset>)")
    g.add_argument("--classes", nargs="+", help="folder dataset: only these class folders")
    g.add_argument("--max-per-class", type=int, help="folder dataset: cap images per class")
    g.add_argument("--val-fraction", type=float, default=d.val_fraction, help="held out per class")
    g.add_argument(
        "--test-fraction", type=float, default=d.test_fraction, help="held out per class"
    )
    g.add_argument("--split-seed", type=int, default=d.split_seed, help="seed of the split only")
    g.add_argument(
        "--augment", choices=("none", "light", "strong"), default=d.augment,
        help="training augmentation",
    )  # fmt: skip
    g.add_argument(
        "--shapes-per-class", type=int, default=d.shapes_per_class, help="shapes dataset size"
    )

    g = p.add_argument_group("model (unset sizes come from config.DATASET_PRESETS[--dataset])")
    g.add_argument("--image-size", type=int, help="input side in pixels")
    g.add_argument("--patch-size", type=int, help="patch side in pixels")
    g.add_argument("--d-model", type=int, help="token width")
    g.add_argument("--n-layers", type=int, help="Transformer blocks")
    g.add_argument("--n-heads", type=int, help="attention heads")
    g.add_argument("--mlp-ratio", type=float, default=m.mlp_ratio, help="MLP width / d_model")
    g.add_argument("--dropout", type=float, default=m.dropout, help="dropout rate")
    g.add_argument(
        "--patch-embed", choices=("linear", "conv"), default=m.patch_embed,
        help="how patches become tokens",
    )  # fmt: skip
    g.add_argument(
        "--pool", choices=("cls", "mean"), default=m.pool, help="what the head classifies"
    )

    g = p.add_argument_group("training")
    g.add_argument("--output-dir", default=t.output_dir, help="where checkpoints go")
    g.add_argument("--epochs", type=int, default=t.epochs, help="passes over the train set")
    g.add_argument("--batch-size", type=int, default=t.batch_size, help="per process / GPU")
    g.add_argument(
        "--grad-accum-steps", type=int, default=t.grad_accum_steps,
        help="micro-batches per optimizer step",
    )  # fmt: skip
    g.add_argument("--lr", type=float, default=t.lr, help="peak learning rate")
    g.add_argument("--weight-decay", type=float, default=t.weight_decay, help="AdamW decay")
    g.add_argument("--warmup-steps", type=int, default=t.warmup_steps, help="linear LR warmup")
    g.add_argument(
        "--label-smoothing", type=float, default=t.label_smoothing, help="0 = plain cross-entropy"
    )
    g.add_argument("--grad-clip", type=float, default=t.grad_clip, help="max grad norm, 0 = off")
    g.add_argument(
        "--precision", choices=("auto", "fp32", "bf16", "fp16"), default=t.precision,
        help="auto = bf16/fp16 on GPU, fp32 on CPU",
    )  # fmt: skip
    g.add_argument("--device", default=t.device, help="auto | cpu | cuda | mps")
    g.add_argument("--num-workers", type=int, default=t.num_workers, help="DataLoader workers")
    g.add_argument("--seed", type=int, default=t.seed, help="init and data order")
    g.add_argument("--log-every", type=int, default=t.log_every, help="steps between log lines")
    g.add_argument(
        "--eval-every", type=int, default=t.eval_every,
        help="steps between validations, 0 = once per epoch",
    )  # fmt: skip
    g.add_argument("--resume", help="checkpoint to continue from (e.g. .../last.pt)")
    g.add_argument(
        "--overfit", type=int,
        help="sanity check: train AND validate on the first N training images, no augmentation",
    )  # fmt: skip
    return p


def configs_from_args(args: argparse.Namespace) -> tuple[DataConfig, dict[str, Any], TrainConfig]:
    """CLI flags -> (DataConfig, model sizes, TrainConfig). num_classes comes from the data."""
    preset = DATASET_PRESETS[args.dataset]
    sizes = {
        k: getattr(args, k) if getattr(args, k) is not None else preset[k] for k in PRESET_FLAGS
    }
    data_config = DataConfig(
        dataset=args.dataset,
        data_dir=args.data_dir or f"data/{args.dataset}",
        classes=tuple(args.classes) if args.classes else None,
        max_per_class=args.max_per_class,
        val_fraction=args.val_fraction,
        test_fraction=args.test_fraction,
        split_seed=args.split_seed,
        image_size=sizes["image_size"],
        augment="none" if args.overfit else args.augment,
        shapes_per_class=args.shapes_per_class,
    )
    model_sizes = {
        **sizes,
        "mlp_ratio": args.mlp_ratio,
        "dropout": 0.0 if args.overfit else args.dropout,
        "patch_embed": args.patch_embed,
        "pool": args.pool,
    }
    train_config = TrainConfig(
        output_dir=args.output_dir,
        epochs=args.epochs,
        # an overfit set smaller than a batch would give zero full batches
        batch_size=min(args.batch_size, args.overfit) if args.overfit else args.batch_size,
        grad_accum_steps=args.grad_accum_steps,
        lr=args.lr,
        weight_decay=0.0 if args.overfit else args.weight_decay,
        warmup_steps=args.warmup_steps,
        label_smoothing=0.0 if args.overfit else args.label_smoothing,
        grad_clip=args.grad_clip,
        precision=args.precision,
        device=args.device,
        num_workers=args.num_workers,
        seed=args.seed,
        log_every=args.log_every,
        eval_every=args.eval_every,
        resume=args.resume,
        overfit=args.overfit,
    )
    return data_config, model_sizes, train_config


def main(argv: Sequence[str] | None = None) -> None:
    args = build_parser().parse_args(argv)
    data_config, model_sizes, train_config = configs_from_args(args)
    info = setup_distributed(train_config.device)
    logging.basicConfig(
        level=logging.INFO if info.is_main else logging.WARNING,
        format="%(asctime)s %(levelname)s %(message)s",
        datefmt="%H:%M:%S",
    )
    try:
        _build_and_train(data_config, model_sizes, train_config, info)
    finally:
        if info.enabled:
            dist.destroy_process_group()


def _build_and_train(
    data_config: DataConfig, model_sizes: dict[str, Any], train_config: TrainConfig, info: DistInfo
) -> TrainState:
    seed_everything(train_config.seed)
    with local_main_first(info):  # CIFAR-10 download happens once per machine
        datasets = build_datasets(data_config)
    train_set, val_set = datasets.train, datasets.val
    if train_config.overfit:
        # Same images for training and validation: the model should reach ~100%.
        # Balanced over classes: the first N items would all be of one class
        # (splits are grouped by class), and 100% on one class proves nothing.
        subset = balanced_subset(train_set.items, train_config.overfit)
        plain = build_transform(data_config, train=False)
        train_set = ImageDataset(subset, train_set.load, plain)
        val_set = ImageDataset(subset, train_set.load, plain)
        counts = Counter(datasets.class_names[label] for _, label in subset)
        logger.info("overfit on %d images: %s", len(subset), dict(sorted(counts.items())))
    logger.info(
        "data: %s | %d classes | train %d / val %d / test %d images",
        data_config.dataset, len(datasets.class_names), len(train_set), len(val_set),
        len(datasets.test),
    )  # fmt: skip

    model_config = ModelConfig(num_classes=len(datasets.class_names), **model_sizes)
    model = VisionTransformer(model_config)
    device = resolve_device(train_config.device)
    train_loader, val_loader = build_dataloaders(train_set, val_set, train_config, device)
    return train(model, train_loader, val_loader, data_config, train_config, datasets.class_names)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/tiny_vit/predict.py
"""Classify images with a trained checkpoint.

    vit-predict --checkpoint checkpoints/run/best.pt --image leaf.jpg [more.jpg ...] [--top-k 3]

From Python:
    from tiny_vit.predict import Predictor
    predictor = Predictor.from_checkpoint("checkpoints/run/best.pt")
    predictor.predict(Image.open("leaf.jpg"))   # [("Tomato___healthy", 0.97), ...]
"""

from __future__ import annotations

import argparse
import json
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from tiny_vit.config import DataConfig, ModelConfig, config_from_dict
from tiny_vit.data import build_transform
from tiny_vit.model import VisionTransformer, attention_rollout
from tiny_vit.train import resolve_device


def load_model(
    path: str | Path, device: torch.device | str = "cpu"
) -> tuple[VisionTransformer, DataConfig, list[str]]:
    """Rebuild a vit-train checkpoint: (model in eval mode, data config, class names)."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model = VisionTransformer(config_from_dict(ModelConfig, checkpoint["model_config"]))
    model.load_state_dict(checkpoint["model"])
    model.to(device).eval()
    data_config = config_from_dict(DataConfig, checkpoint["data_config"])
    return model, data_config, list(checkpoint["class_names"])


@dataclass
class Predictor:
    """A model plus everything needed to feed it raw images."""

    model: VisionTransformer
    data_config: DataConfig
    class_names: list[str]
    device: torch.device

    @classmethod
    def from_checkpoint(cls, path: str | Path, device: str = "auto") -> Predictor:
        resolved = resolve_device(device)
        model, data_config, class_names = load_model(path, resolved)
        return cls(model, data_config, class_names, resolved)

    def preprocess(self, images: Sequence[Image.Image]) -> torch.Tensor:
        """PIL images -> (B, 3, S, S) batch, exactly like the validation images."""
        transform = build_transform(self.data_config, train=False)
        return torch.stack([transform(image.convert("RGB")) for image in images]).to(self.device)

    @torch.inference_mode()
    def probabilities(self, images: Sequence[Image.Image]) -> torch.Tensor:
        """(B, num_classes) class probabilities."""
        return self.model(self.preprocess(images)).float().softmax(dim=-1).cpu()

    def predict(self, image: Image.Image, top_k: int = 5) -> list[tuple[str, float]]:
        """[(class name, probability)], most likely first."""
        probs = self.probabilities([image])[0]
        values, indices = probs.topk(min(top_k, len(self.class_names)))
        return [
            (self.class_names[i], float(v)) for v, i in zip(values, indices.tolist(), strict=True)
        ]

    def attention_heatmap(self, image: Image.Image) -> np.ndarray:
        """Attention rollout as an (S/P, S/P) array in [0, 1]: where the model looked."""
        attentions = self.model.attention_maps(self.preprocess([image]))
        grid = self.data_config.image_size // self.model.config.patch_size
        heat = attention_rollout(attentions)[0].reshape(grid, grid).float().cpu().numpy()
        return (heat - heat.min()) / max(float(heat.max() - heat.min()), 1e-12)


def overlay_heatmap(image: Image.Image, heat: np.ndarray, alpha: float = 0.5) -> Image.Image:
    """Blend a heatmap (red = high attention) over the image, at the image's size."""
    image = image.convert("RGB")
    mask = Image.fromarray((heat * 255).astype(np.uint8)).resize(image.size, Image.BICUBIC)
    m = np.asarray(mask, dtype=np.float32)[..., None] / 255.0
    red = np.zeros_like(np.asarray(image, dtype=np.float32))
    red[..., 0] = 255.0
    pixels = np.asarray(image, dtype=np.float32)
    blended = pixels * (1 - alpha * m) + red * (alpha * m)
    return Image.fromarray(blended.clip(0, 255).astype(np.uint8))


def main(argv: Sequence[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Classify images with a trained Tiny-ViT checkpoint.")
    p.add_argument("--checkpoint", required=True, type=Path)
    p.add_argument("--image", required=True, nargs="+", type=Path)
    p.add_argument("--top-k", type=int, default=3)
    p.add_argument("--device", default="auto")
    p.add_argument("--json", action="store_true", help="print JSON instead of text")
    args = p.parse_args(argv)

    predictor = Predictor.from_checkpoint(args.checkpoint, args.device)
    results = {}
    for path in args.image:
        with Image.open(path) as image:
            results[str(path)] = predictor.predict(image, args.top_k)
    if args.json:
        print(
            json.dumps(
                {k: [{"class": c, "probability": p} for c, p in v] for k, v in results.items()},
                indent=2,
            )
        )
        return
    for path, ranked in results.items():
        print(path)
        for name, prob in ranked:
            print(f"  {prob:7.2%}  {name}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/tiny_vit/evaluate.py
"""Score a checkpoint on its held-out test (or validation) split.

    vit-evaluate --checkpoint checkpoints/run/best.pt [--split test] [--examples docs/examples/run]

Prints overall accuracy, per-class recall / precision and the confusion matrix.
The split is rebuilt from the checkpoint's DataConfig (same split_seed), so it
contains exactly the images the model never trained on. With --examples, it
also writes thumbnails and a Markdown table (image, prediction, truth) of
images spread evenly over the split: real, not hand-picked, outputs.
"""

from __future__ import annotations

import argparse
import json
from collections.abc import Sequence
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_pil_image

from tiny_vit.data import build_datasets, denormalize
from tiny_vit.predict import Predictor


@torch.inference_mode()
def confusion_matrix(predictor: Predictor, loader: DataLoader) -> torch.Tensor:
    """(num_classes, num_classes) counts; row = true class, column = predicted class."""
    n = len(predictor.class_names)
    matrix = torch.zeros(n, n, dtype=torch.long)
    for images, labels in loader:
        predictions = predictor.model(images.to(predictor.device)).argmax(dim=-1).cpu()
        matrix += torch.bincount(labels * n + predictions, minlength=n * n).view(n, n)
    return matrix


def summarize(matrix: torch.Tensor, class_names: Sequence[str]) -> dict:
    """Accuracy plus per-class recall (how many of class c we found) and precision
    (how many predictions of c were right)."""
    correct = matrix.diag().float()
    per_class = {
        name: {
            "recall": float(correct[i] / max(int(matrix[i].sum()), 1)),
            "precision": float(correct[i] / max(int(matrix[:, i].sum()), 1)),
            "support": int(matrix[i].sum()),
        }
        for i, name in enumerate(class_names)
    }
    return {
        "accuracy": float(correct.sum() / max(int(matrix.sum()), 1)),
        "images": int(matrix.sum()),
        "per_class": per_class,
        "confusion_matrix": matrix.tolist(),
    }


def format_report(report: dict, class_names: Sequence[str]) -> str:
    width = max(len(n) for n in class_names)
    lines = [
        f"accuracy {report['accuracy']:.2%} on {report['images']} images",
        "",
        f"{'class':<{width}}  recall  precision  images",
    ]
    for name, row in report["per_class"].items():
        lines.append(
            f"{name:<{width}}  {row['recall']:6.1%}  {row['precision']:9.1%}  {row['support']:6d}"
        )
    lines += ["", "confusion matrix (row = true class, column = predicted):"]
    lines += ["  " + " ".join(f"{v:5d}" for v in row) for row in report["confusion_matrix"]]
    return "\n".join(lines)


def write_examples(predictor: Predictor, dataset, out_dir: Path, count: int) -> Path:
    """Thumbnails + README.md table for `count` images spread evenly over the dataset."""
    out_dir.mkdir(parents=True, exist_ok=True)
    step = max(1, len(dataset) // count)
    rows = ["| image | prediction | confidence | truth | correct |", "|:---:|---|---:|---|:---:|"]
    for k, index in enumerate(range(0, len(dataset), step)[:count]):
        pixels, label = dataset[index]
        image = to_pil_image(denormalize(pixels, predictor.data_config))
        name, prob = predictor.predict(image, top_k=1)[0]
        truth = predictor.class_names[label]
        thumb = image.resize((96, 96), Image.NEAREST if image.width < 96 else Image.BICUBIC)
        thumb.save(out_dir / f"{k:02d}.jpg", quality=92)
        rows.append(
            f'| <img src="{k:02d}.jpg" width="72"> | {name} | {prob:.0%} | {truth} | '
            f"{'yes' if name == truth else '**no**'} |"
        )
    readme = out_dir / "README.md"
    readme.write_text("\n".join(rows) + "\n")
    return readme


def main(argv: Sequence[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Evaluate a Tiny-ViT checkpoint on a held-out split.")
    p.add_argument("--checkpoint", required=True, type=Path)
    p.add_argument("--split", choices=("test", "val"), default="test")
    p.add_argument("--data-dir", help="override the dataset folder stored in the checkpoint")
    p.add_argument("--batch-size", type=int, default=128)
    p.add_argument("--num-workers", type=int, default=2)
    p.add_argument("--device", default="auto")
    p.add_argument("--report", type=Path, help="also save the numbers as JSON")
    p.add_argument("--examples", type=Path, help="write an example table to this folder")
    p.add_argument("--num-examples", type=int, default=12)
    args = p.parse_args(argv)

    predictor = Predictor.from_checkpoint(args.checkpoint, args.device)
    if args.data_dir:
        predictor.data_config.data_dir = args.data_dir
    datasets = build_datasets(predictor.data_config)
    dataset = datasets.test if args.split == "test" else datasets.val
    if datasets.class_names != predictor.class_names:
        raise SystemExit(
            f"the data's classes {datasets.class_names} differ from the checkpoint's "
            f"{predictor.class_names}"
        )
    loader = DataLoader(dataset, batch_size=args.batch_size, num_workers=args.num_workers)
    report = summarize(confusion_matrix(predictor, loader), predictor.class_names)
    report["split"] = args.split
    print(format_report(report, predictor.class_names))
    if args.report:
        args.report.parent.mkdir(parents=True, exist_ok=True)
        args.report.write_text(json.dumps(report, indent=2))
    if args.examples:
        readme = write_examples(predictor, dataset, args.examples, args.num_examples)
        print(f"\nexamples written to {readme}")


if __name__ == "__main__":
    main()

In [ ]:
import sys

sys.path.insert(0, "src")
for name in [m for m in sys.modules if m.startswith("tiny_vit")]:
    del sys.modules[name]  # pick up the files just written if this cell is re-run
import tiny_vit.train as T

print("tiny_vit ready")

## 2. Settings

| `DATASET` | Images | What for |
|---|---|---|
| `"shapes"` | 12 000 generated shapes, 6 classes, no download | prove the model learns from pixels (minutes) |
| `"cifar10"` | 60 000 photos 32×32, 10 classes, 170 MB | the standard small benchmark |
| `"plantvillage"` | ~15 000 leaf photos, 12 tomato / potato classes, ~0.3 GB | the original task of this project |

`SMOKE_TEST = True` runs a tiny version to check the whole notebook in a few
minutes. Keep it `False` for real training.

In [ ]:
import dataclasses
import subprocess
from pathlib import Path

import torch

from tiny_vit.config import DATASET_PRESETS, PLANTVILLAGE_TOMATO_POTATO, DataConfig, TrainConfig

SMOKE_TEST = False          # True = tiny quick check of the whole notebook
DATASET = "plantvillage"    # "shapes" | "cifar10" | "plantvillage"
USE_GOOGLE_DRIVE = False    # Colab only: keep checkpoints on Drive (survive disconnects)
RESUME = True               # continue from last.pt in OUTPUT_DIR if it exists

# --- where checkpoints go --------------------------------------------------------
ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ
if ON_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path(f"/content/drive/MyDrive/tiny-vit/{DATASET}")
elif ON_KAGGLE:
    OUTPUT_DIR = Path(f"/kaggle/working/checkpoints/{DATASET}")
else:
    OUTPUT_DIR = Path(f"checkpoints/{DATASET}")

# --- data ----------------------------------------------------------------------
source = "folder" if DATASET == "plantvillage" else DATASET
if DATASET == "plantvillage":
    data_dir = "data/plantvillage_hf/color"
    if not Path(data_dir).exists():
        subprocess.check_call([
            "hf", "download", "leoho36/plant_village_dataset", "--repo-type", "dataset",
            "--include", "color/Tomato*", "--include", "color/Potato*",
            "--exclude", "*Yellow_Leaf_Curl*",  # not one of the 12 classes
            "--local-dir", "data/plantvillage_hf", "--max-workers", "16",
        ])
    classes = PLANTVILLAGE_TOMATO_POTATO
else:
    data_dir, classes = f"data/{DATASET}", None

preset = DATASET_PRESETS[source]
data_config = DataConfig(
    dataset=source, data_dir=data_dir, classes=classes, image_size=preset["image_size"],
    augment="strong" if DATASET != "shapes" else "light",
)
model_size = {k: preset[k] for k in ("image_size", "patch_size", "d_model", "n_layers", "n_heads")}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    precision = "bf16" if props.major >= 8 else "fp16"  # native bf16 needs compute capability 8+
    print(f"GPU: {props.name} ({props.total_memory / 1e9:.0f} GB)")
else:
    precision = "fp32"
    print("No GPU: enable one in the notebook settings (fine only for SMOKE_TEST / shapes).")

epochs = {"shapes": 15, "cifar10": 100, "plantvillage": 60}[DATASET]
train_config = TrainConfig(
    output_dir=str(OUTPUT_DIR), epochs=epochs, batch_size=128, lr=1e-3, warmup_steps=500,
    precision=precision, device=device.type, num_workers=min(4, os.cpu_count() or 1),
    log_every=25,
)
if SMOKE_TEST:
    OUTPUT_DIR = OUTPUT_DIR.parent / "smoke-test"  # never mixed with a real run
    data_config = dataclasses.replace(data_config, max_per_class=20, shapes_per_class=20)
    model_size.update(d_model=64, n_layers=2, n_heads=4)
    train_config = dataclasses.replace(
        train_config, output_dir=str(OUTPUT_DIR), epochs=3, batch_size=16, warmup_steps=2,
        log_every=2, num_workers=0,
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if RESUME and (OUTPUT_DIR / T.LAST_CHECKPOINT).exists():
    train_config = dataclasses.replace(train_config, resume=str(OUTPUT_DIR / T.LAST_CHECKPOINT))

for name, cfg in [("DataConfig", data_config), ("TrainConfig", train_config)]:
    print(f"\n{name}:")
    for key, value in dataclasses.asdict(cfg).items():
        print(f"  {key:18} {value}")
print(f"\nmodel: {model_size}")

## 3. Data

Stratified train / validation / test split (same class balance in each). The
test split is never used for training or for picking the best checkpoint.

In [ ]:
import logging

import matplotlib.pyplot as plt

from tiny_vit.data import build_datasets, denormalize

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S",
                    force=True)
T.seed_everything(train_config.seed)
datasets = build_datasets(data_config)
class_names = datasets.class_names
print(f"{len(class_names)} classes | train {len(datasets.train):,} | "
      f"val {len(datasets.val):,} | test {len(datasets.test):,}")

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, i in zip(axes, range(0, len(datasets.val), max(1, len(datasets.val) // 6)), strict=False):
    pixels, label = datasets.val[i]
    ax.imshow(denormalize(pixels, data_config).permute(1, 2, 0))
    ax.set_title(class_names[label], fontsize=8)
    ax.axis("off")
plt.show()

train_loader, val_loader = T.build_dataloaders(datasets.train, datasets.val, train_config, device)
print(f"{len(train_loader):,} train batches per epoch")

## 4. Model

In [ ]:
import math

import torch.nn.functional as F

from tiny_vit.config import ModelConfig
from tiny_vit.model import VisionTransformer

model_config = ModelConfig(num_classes=len(class_names), **model_size)
model = VisionTransformer(model_config).to(device)
print(f"parameters: {model.num_parameters() / 1e6:.2f}M | tokens per image: "
      f"{model_config.num_patches} patches + [CLS]")

# Sanity check: the untrained model guesses uniformly, loss = ln(num_classes).
images, labels = next(iter(val_loader))
with torch.no_grad():
    loss = F.cross_entropy(model(images.to(device)), labels.to(device))
print(f"initial loss {loss.item():.3f} (uniform guessing = {math.log(len(class_names)):.3f})")

## 5. Training with a live dashboard

Same loop as `vit-train`: AdamW, warmup + cosine schedule, mixed precision,
gradient clipping, label smoothing. After every epoch it validates and saves
`last.pt`, plus `best.pt` when the validation accuracy improves.

**Interrupted, or hit the session limit?** Run the notebook again with
`RESUME = True`: it continues from `last.pt` (same step, same data order).

In [ ]:
from IPython.display import clear_output

history = {"train": [], "val": []}


def draw_dashboard():
    train, val = history["train"], history["val"]
    clear_output(wait=True)
    fig, (ax_loss, ax_acc, ax_lr) = plt.subplots(1, 3, figsize=(16, 4))
    ax_loss.plot([r["step"] for r in train], [r["loss"] for r in train], label="train", alpha=0.8)
    ax_loss.plot([r["step"] for r in val], [r["loss"] for r in val], "o-", label="validation")
    ax_loss.set_title("loss")
    ax_acc.plot([r["step"] for r in train], [r["accuracy"] for r in train], label="train", alpha=0.8)
    ax_acc.plot([r["step"] for r in val], [r["accuracy"] for r in val], "o-", label="validation")
    ax_acc.set_title("accuracy")
    ax_lr.plot([r["step"] for r in train], [r["lr"] for r in train])
    ax_lr.set_title("learning rate")
    for ax in (ax_loss, ax_acc, ax_lr):
        ax.set_xlabel("optimizer step")
        ax.grid(alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend()
    plt.show()
    if train:
        last = train[-1]
        print(f"step {last['step']:,}/{last['total_steps']:,} | epoch {last['epoch'] + 1} | "
              f"loss {last['loss']:.4f} | {last['images_per_sec']:.0f} img/s")
    if val:
        print(f"validation accuracy {val[-1]['accuracy']:.2%} (best "
              f"{max(r['accuracy'] for r in val):.2%})")


def on_metrics(event):
    history[event["kind"]].append(event)
    draw_dashboard()


state = T.train(model, train_loader, val_loader, data_config, train_config, class_names,
                callback=on_metrics)
print(f"Training finished. Best validation accuracy {state.best_val_accuracy:.2%}")

## 6. Results on the test split

The best checkpoint on images it never saw: accuracy, per-class recall /
precision, confusion matrix, and a table of real examples (`examples/README.md`)
ready to commit to GitHub.

In [ ]:
from tiny_vit import evaluate

evaluate.main(["--checkpoint", str(OUTPUT_DIR / T.BEST_CHECKPOINT), "--split", "test",
               "--report", str(OUTPUT_DIR / "test_report.json"),
               "--examples", str(OUTPUT_DIR / "examples"), "--num-examples", "12"])

In [ ]:
from PIL import Image

from tiny_vit.predict import Predictor, overlay_heatmap

predictor = Predictor.from_checkpoint(OUTPUT_DIR / T.BEST_CHECKPOINT, device.type)
fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
step = max(1, len(datasets.test) // 6)
for col, i in enumerate(range(0, len(datasets.test), step)[:6]):
    pixels, label = datasets.test[i]
    image = Image.fromarray((denormalize(pixels, data_config).permute(1, 2, 0) * 255).byte().numpy())
    name, prob = predictor.predict(image, top_k=1)[0]
    axes[0, col].imshow(image)
    axes[0, col].set_title(f"{name} ({prob:.0%})\ntrue: {class_names[label]}", fontsize=7)
    axes[1, col].imshow(overlay_heatmap(image, predictor.attention_heatmap(image)))
    axes[1, col].set_title("attention rollout", fontsize=7)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Download the trained model

Zips `best.pt`, the test report and the examples (`last.pt` is left out: it is
only needed to resume). Use it with the repo:

```bash
unzip tiny-vit-run.zip -d checkpoints/run
vit-predict --checkpoint checkpoints/run/best.pt --image leaf.jpg
streamlit run app/streamlit_app.py -- --checkpoint checkpoints/run/best.pt
```

In [ ]:
import shutil
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    bundle = Path(tmp) / "bundle"
    bundle.mkdir()
    for name in [T.BEST_CHECKPOINT, "test_report.json"]:
        if (OUTPUT_DIR / name).exists():
            shutil.copy(OUTPUT_DIR / name, bundle / name)
    if (OUTPUT_DIR / "examples").exists():
        shutil.copytree(OUTPUT_DIR / "examples", bundle / "examples")
    archive = shutil.make_archive(str(OUTPUT_DIR.parent / "tiny-vit-run"), "zip", bundle)
print("created", archive)

if ON_COLAB:
    from google.colab import files

    files.download(archive)
elif ON_KAGGLE:
    print("Kaggle: click 'Save Version', then download it from the notebook's Output tab.")